# Quickstart

This tutorial shows how to open a Planet Tanager hyperspectral data cube
and create basic visualizations using `xarray-hyperspectral`.

## Download a Tanager file

For this tutorial we'll download a surface reflectance product from the [Tanager Open Data Catalog](https://www.planet.com/data/stac/browser/).

In [ ]:
import urllib.request, os

url = (
    "https://storage.googleapis.com/open-cogs/planet-stac/"
    "tanager1-release2-core-imagery/basic_sr_hdf5/"
    "20250926_092059_95_4001_basic_sr_hdf5.h5"
)
filename = url.rsplit("/", 1)[-1]
if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

## Open the file

After `pip install xarray-hyperspectral`, the `engine="tanager"` backend is
automatically registered with xarray. No extra imports needed.

In [ ]:
import xarray as xr

ds = xr.open_dataset(filename, engine="tanager")
ds

## Plot a single band

Tanager captures 426 wavelength bands from approximately 380 to 2500 nm.
We can select any band by wavelength using xarray's `.sel()` method.
The `method="nearest"` argument snaps to the closest available wavelength.

In [ ]:
band = ds["reflectance"].sel(wavelength=850, method="nearest")
plot = band.plot(robust=True, cmap="viridis", figsize=(8, 8), aspect="equal")
plot.axes.set_title("Surface reflectance at 850 nm");

## RGB composite

With hundreds of bands available, we can pick three wavelengths near
red (650 nm), green (550 nm), and blue (460 nm) to build a true-color image.

In [ ]:
# Select bands closest to red, green, and blue wavelengths
rgb = (
    ds["reflectance"]
    .sel(wavelength=[650, 550, 460], method="nearest")
    .transpose(..., "band")
)

# Scale reflectance values for better visualization
rgb = (rgb / 0.4).clip(0, 1)

plot = rgb.plot.imshow(figsize=(8, 8))
plot.axes.set_aspect("equal")
plot.axes.set_title("True-color composite (R=650, G=550, B=460 nm)")
plot.axes.set_axis_off()

## Plot a spectrum

The real power of hyperspectral data is the full spectrum at every pixel.
Here we extract the reflectance across all 426 bands at a single pixel.

In [ ]:
spectrum = ds["reflectance"].sel(crosstrack=50, alongtrack=100)
spectrum = spectrum.where(ds.good_wavelengths)

uncertainty = ds["reflectance_uncertainty"].sel(crosstrack=50, alongtrack=100)
uncertainty = uncertainty.where(ds.good_wavelengths)

plot = spectrum.plot(x="wavelength", figsize=(10, 4), label="Reflectance")
ax = plot[0].axes
ax.fill_between(
    spectrum.wavelength,
    spectrum - uncertainty,
    spectrum + uncertainty,
    alpha=0.5,
    label="±1σ uncertainty",
)
ax.set_title("Reflectance spectrum at a single pixel");
ax.legend();